# CS-781 Phase 1 — Extra models (notebook C): masked diffusion + mixture of experts

Two more architectures, trained in parallel (one per GPU), using the **same data-prep output** as `phase1-train`.

| arch | Model | One line |
|---|---|---|
| `diffusion` | **LexDiffusionLM** | Masked-diffusion LM (MDLM 2024, LLaDA 2025): fills all ~200 continuation slots over several rounds, most-confident guesses first. Bidirectional, so also an encoder for Phases 2–5. |
| `moe` | **LexMoE** | Decoder whose feed-forward part is a Mixture of Experts (Switch 2022, OLMoE 2024): a router sends each token to one of several experts. Research question: do experts specialise by court type? |

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · secret `Capstone_Phase_1` on · **Add Input → Your Work → Notebooks → `phase1-data-prep`**.

**How to use**
1. **Now, in a CPU session** (session 2 keeps the GPUs): run 1, X3, X4, X5 (your `models_x.py`), X5b (sizes) and X5c (CPU smoke test).
2. **When session 2 has finished:** switch to **GPU T4 ×2**, then **Save Version → Save & Run All**. X6 re-checks on GPU, X8 trains both models (~`hours_per_model`), then generates and writes `runs/<arch>/submission_<arch>.zip`.
3. Afterwards: X10 (scores), X11 (zip check), X12 (expert routing by court type, for the report).

**Timing:** `hours_per_model` 4.0 → about 5–5.5 h in total (the diffusion model generates faster; LexMoE generates at about the decoder's speed). Leave several hours before the 11:59 PM deadline for Codabench's queue.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU (on a CPU session this shows 0 GPUs — that is fine for steps X3–X5c)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## X3. Configuration
`train` and `gen` are the same values as your `phase1-train` run. Model sizes are **your** decision (X5b shows the counts; every model must be 50M–300M parameters). For LexMoE the 300M limit counts **all** experts; X5b also shows the *active* parameters per token. `diff_gen` are the diffusion sampling settings, also your call. The default is MDLM's own sampler (random filling order, temperature 1.0): in testing, greedy filling (temperature 0, LLaDA's choice for 8B models) made a small model fill every slot with punctuation.

In [ ]:
import os, json, shutil
from huggingface_hub import hf_hub_download

SESSION_ARCHS = ["diffusion", "moe"]     # one model per GPU, trained in parallel
ALL_ARCHS = ["diffusion", "moe"]

CONFIG = {
    "seed": 42,
    "hours_per_model": 4.0,               # training wall-clock per model (YOUR call; see the timing note above)
    "train": {                            # same starter-notebook defaults as phase1-train (YOUR call)
        "block_size": 1024, "batch_size": 8, "grad_accum": 2,
        "lr": 3e-4, "min_lr": 0.0, "warmup_steps": 500, "weight_decay": 0.1,
        "betas": [0.9, 0.999], "grad_clip": 1.0, "log_every": 50, "eval_every": 500,
    },
    "moe_aux_coef": 0.01,                 # weight of the load-balancing loss (Switch Transformer used 0.01)
    "models": {                           # YOUR sizes. Keys = the arguments of your classes in models_x.py
        "diffusion": {"d_model": 672, "n_layers": 8, "n_heads": 8, "max_len": 1024, "dropout": 0.1},
        "moe":       {"d_model": 672, "n_layers": 8, "n_heads": 8, "d_ff": 2688, "n_experts": 4,
                      "max_len": 1024, "dropout": 0.1},
    },
    "gen": {                              # left-to-right decoding for "moe" (your phase1-train choices)
        "temperature": 0.7, "top_k": 0, "top_p": 0.9, "rep_penalty": 1.1,
        "target_words": 128, "max_new": 320, "batch_size": 16,
    },
    "diff_gen": {                         # diffusion sampling (YOUR call). Default = MDLM's ancestral sampler
        "gen_tokens": 200,                # [MASK] slots after the prompt (~164 words at 1.217 tokens/word)
        "steps": 64,                      # total filling rounds (more = slower, usually better)
        "blocks": 4,                      # filled left to right; 1 = whole continuation at once
        "temperature": 1.0,               # 1.0 = sample from the model (MDLM); 0 = top guess (LLaDA; collapses to
                                          #   punctuation in small models — see the note above)
        "remask": "random",               # which guesses to keep each round: "random" (MDLM) or "confidence" (LLaDA)
        "target_words": 128, "batch_size": 16,
    },
}
json.dump(CONFIG, open("/kaggle/working/run_config_x.json", "w"), indent=1)

test_src = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json",
                           repo_type="dataset", token=HF_TOKEN)
shutil.copy(test_src, "/kaggle/working/test.json")
print("Config written; test prompts:", len(json.load(open("/kaggle/working/test.json"))))

## X4. Pipeline code (same as phase1-train's `lm_common.py`, plus the diffusion loss/sampler and the MoE load-balancing hook)

In [ ]:
%%writefile lm_common_x.py
"""Pipeline for the extra models: same as lm_common.py from phase1-train, plus
   * kind "diffusion": masked-diffusion training loss + LLaDA-style sampler
   * models that expose .aux_loss (mixture of experts): load-balancing loss is added during training."""
import glob, json, math, os, random, sys, zipfile
import numpy as np, torch, torch.nn.functional as F

def find_input(name):
    for root in ("/kaggle/working", "/kaggle/input"):
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found — is the phase1-data-prep output added as input?")

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

WORD_MARK = "".join(chr(b) for b in "\u2581".encode("utf-8"))   # how '▁' looks inside your byte-level tokens

class Tok:
    def __init__(self, path=None):
        self.bpe = bpe_tok.load_bpe(path or find_input("bpe.json"))
        self.enc = bpe_tok.FastEncoder(self.bpe)
        self.itos, self.V = self.bpe.itos, len(self.bpe.itos)
        self.pad, self.unk, self.bos, self.eos = (self.bpe.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
        self.mask = self.unk          # [MASK] for the diffusion model: <unk> never occurs in the data (0% unk)
        self.word_start = torch.tensor([t.startswith(WORD_MARK) for t in self.itos])
    def encode(self, text):
        return self.enc.encode(text)
    def decode(self, ids):
        return bpe_tok.detokenize([self.itos[i] for i in ids if i >= 4])

def get_batch(data, batch_size, block, device, rng=np.random):
    ix = rng.randint(0, len(data) - block - 1, batch_size)
    x = torch.from_numpy(np.stack([data[i:i + block + 1] for i in ix]).astype(np.int64))
    if str(device).startswith("cuda"):
        x = x.pin_memory()
    return x.to(device, non_blocking=True)

def diffusion_loss(model, x, mask_id, rng=random, t_min=1e-3, min_target=128):
    """Masked-diffusion objective (MDLM / LLaDA). A clean prefix of random length P plays the prompt; every
    later token is replaced by [MASK] with probability t (one t per sequence). The model predicts the hidden
    tokens; each hidden token's loss is weighted by 1/t, normalised by the target length.
    t uses MDLM's low-discrepancy sampler: the B sequences of a batch get evenly spread t values
    (one per stratum [i/B, (i+1)/B)), which lowers the variance of the loss compared with independent draws."""
    B, T = x.shape
    P = rng.randint(0, T - min(min_target, T // 2))   # clean prompt length; at least min_target tokens to predict
    u = torch.rand(1, device=x.device)
    t = ((u + torch.arange(B, device=x.device) / B) % 1.0).view(B, 1) * (1 - t_min) + t_min
    hide = torch.rand(B, T, device=x.device) < t
    hide[:, :P] = False
    noisy = torch.where(hide, torch.full_like(x, mask_id), x)
    logits = model(noisy)
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), x.reshape(-1), reduction="none").view(B, T)
    return ((ce * hide / t).sum(1) / (T - P)).mean()

def compute_loss(model, seq, bos_id, rng=random, mask_id=None, aux_coef=0.0):
    """seq: (B, block+1). Training loss for model.kind (+ aux_coef * model.aux_loss when the model has one)."""
    if model.kind == "diffusion":
        return diffusion_loss(model, seq[:, :-1], mask_id, rng)
    if model.kind == "causal":
        logits, target = model(seq[:, :-1]), seq[:, 1:]
    elif model.kind == "prefix":
        T = seq.size(1) - 1
        P = rng.randint(T // 4, 3 * T // 4)
        logits = model(seq[:, :-1], prefix_len=P)[:, P - 1:]
        target = seq[:, P:]
    elif model.kind == "encdec":
        L = seq.size(1)
        S = rng.randint(max(16, L - model.max_tgt_len), min(model.max_len, L - 16))
        src, tgt = seq[:, :S], seq[:, S:]
        tgt_in = torch.cat([torch.full_like(tgt[:, :1], bos_id), tgt[:, :-1]], 1)
        logits, target = model(src, tgt_in), tgt
    else:
        raise ValueError(model.kind)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), target.reshape(-1))
    aux = getattr(model, "aux_loss", None)
    if aux_coef and aux is not None:
        loss = loss + aux_coef * aux
    return loss

@torch.no_grad()
def eval_loss(model, data, batch_size, block, device, bos_id, mask_id=None, n_batches=20):
    """Validation loss WITHOUT the auxiliary loss. Uses fixed random seeds so every evaluation sees the
    same batches (and, for diffusion, the same masks), without disturbing the training random state."""
    model.eval()
    rng, prng, losses = np.random.RandomState(1234), random.Random(1234), []
    devs = [torch.device(device).index or 0] if str(device).startswith("cuda") else []
    with torch.random.fork_rng(devices=devs):
        torch.manual_seed(1234)
        for _ in range(n_batches):
            seq = get_batch(data, batch_size, block, device, rng)
            with torch.autocast("cuda", dtype=torch.float16):
                losses.append(compute_loss(model, seq, bos_id, prng, mask_id=mask_id).item())
    model.train()
    return float(np.mean(losses))

def sample_next(logits, generated, temperature, top_k, top_p, rep_penalty, banned):
    logits = logits.float().clone()
    logits[:, banned] = -float("inf")
    if rep_penalty != 1.0 and generated.size(1) > 0:
        prev = torch.gather(logits, 1, generated)
        logits.scatter_(1, generated, torch.where(prev > 0, prev / rep_penalty, prev * rep_penalty))
    if temperature <= 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        sorted_logits = sorted_logits.masked_fill(probs.cumsum(-1) - probs > top_p, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)

@torch.no_grad()
def generate(model, tok, prompts, device, temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.0,
             target_words=128, max_new=320, batch_size=16, seed=0, log_every=10):
    """Left-to-right generation (causal / prefix / encdec models). Same as in phase1-train."""
    torch.manual_seed(seed)
    model.eval()
    ids = [tok.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    ws = tok.word_start.to(device)
    banned = torch.tensor([tok.pad, tok.unk, tok.bos, tok.eos], device=device)
    if model.kind == "encdec":
        max_prompt, max_new = model.max_len, min(max_new, model.max_tgt_len - 1)
    else:
        max_prompt = model.max_len - max_new
    out = [""] * len(ids)
    for bi, b0 in enumerate(range(0, len(order), batch_size)):
        bidx = order[b0:b0 + batch_size]
        L = min(min(len(ids[i]) for i in bidx), max_prompt)
        prompt = torch.tensor([ids[i][-L:] for i in bidx], device=device)
        B = len(bidx)
        gen = torch.empty(B, 0, dtype=torch.long, device=device)
        for _ in range(max_new):
            with torch.autocast("cuda", dtype=torch.float16):
                if model.kind == "encdec":
                    tgt = torch.cat([torch.full((B, 1), tok.bos, device=device), gen], 1)
                    logits = model(prompt, tgt)[:, -1]
                elif model.kind == "prefix":
                    logits = model(torch.cat([prompt, gen], 1), prefix_len=L)[:, -1]
                else:
                    logits = model(torch.cat([prompt, gen], 1))[:, -1]
            nxt = sample_next(logits, gen, temperature, top_k, top_p, rep_penalty, banned)
            gen = torch.cat([gen, nxt[:, None]], 1)
            if (ws[gen].sum(1) > target_words).all():
                break
        for j, i in enumerate(bidx):
            words = tok.decode(gen[j].tolist()).split()[:target_words]
            out[i] = " ".join(words) if words else "the"
        if log_every and bi % log_every == 0:
            print(f"  generated {min(b0 + batch_size, len(order))}/{len(order)}", flush=True)
    return out

@torch.no_grad()
def generate_diffusion(model, tok, prompts, device, gen_tokens=200, steps=64, blocks=4, temperature=1.0,
                       remask="random", target_words=128, batch_size=16, seed=0, log_every=10):
    """Masked-diffusion sampling with semi-autoregressive blocks.
    After the prompt, `gen_tokens` [MASK] slots are appended and split into `blocks` blocks filled left to
    right. Each block gets steps/blocks rounds; in each round the model guesses every hidden slot of the block
    and keeps k of the guesses, so the block is complete after its last round. Which guesses are kept:
      remask="confidence" — the most confident ones (LLaDA). Weak models may favour punctuation.
      remask="random"     — random hidden slots (MDLM's original ancestral sampler)."""
    assert gen_tokens % blocks == 0 and steps % blocks == 0, "gen_tokens and steps must divide by blocks"
    torch.manual_seed(seed)
    model.eval()
    ids = [tok.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    banned = torch.tensor([tok.pad, tok.unk, tok.bos, tok.eos], device=device)
    max_prompt = model.max_len - gen_tokens
    blk, per_block = gen_tokens // blocks, steps // blocks
    out = [""] * len(ids)
    for bi, b0 in enumerate(range(0, len(order), batch_size)):
        bidx = order[b0:b0 + batch_size]
        L = min(min(len(ids[i]) for i in bidx), max_prompt)
        prompt = torch.tensor([ids[i][-L:] for i in bidx], device=device)
        B = len(bidx)
        x = torch.cat([prompt, torch.full((B, gen_tokens), tok.mask, dtype=torch.long, device=device)], 1)
        for b in range(blocks):
            lo, hi = L + b * blk, L + (b + 1) * blk
            for s in range(per_block):
                hidden = x[:, lo:hi] == tok.mask
                n_left = int(hidden[0].sum())                # every row has the same number left
                if n_left == 0:
                    break
                k = math.ceil(n_left / (per_block - s))
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(x)[:, lo:hi].float()
                logits[:, :, banned] = -float("inf")
                if temperature > 0:
                    probs = (logits / temperature).softmax(-1)
                    pred = torch.multinomial(probs.reshape(-1, probs.size(-1)), 1).view(B, -1)
                else:
                    pred = logits.argmax(-1)
                if remask == "random":
                    conf = torch.rand(pred.shape, device=device)
                else:
                    conf = logits.softmax(-1).gather(-1, pred[..., None]).squeeze(-1)
                conf = conf.masked_fill(~hidden, -1.0)       # never re-pick slots that are already filled
                pick = conf.topk(k, dim=1).indices
                x[:, lo:hi] = x[:, lo:hi].scatter(1, pick, pred.gather(1, pick))
        for j, i in enumerate(bidx):
            words = tok.decode(x[j, L:].tolist()).split()[:target_words]
            out[i] = " ".join(words) if words else "the"
        if log_every and bi % log_every == 0:
            print(f"  generated {min(b0 + batch_size, len(order))}/{len(order)}", flush=True)
    return out

def generate_any(model, tok, prompts, device, C):
    """Picks the right generator for the model kind, with the matching settings from the config."""
    if model.kind == "diffusion":
        return generate_diffusion(model, tok, prompts, device, **C["diff_gen"])
    return generate(model, tok, prompts, device, **C["gen"])

def write_submission(preds, ids, json_path, zip_path=None):
    missing = [i for i in ids if i not in preds]
    assert not missing, f"{len(missing)} ids missing"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({i: {"text": preds[i]} for i in ids}, f, ensure_ascii=False, indent=2)
    if zip_path:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(json_path, arcname="prediction.json")

## X5. `models_x.py` — YOUR two architectures
Same rules as before (§17.5): the stub gives the contract and plain-English steps; you write the lines. Start by pasting your three helpers from `phase1-train` cell B5 into PART 0.

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## X5b. Size calculator (instant, works on CPU)

In [ ]:
# Size calculator: instant, no GPU memory. Edit sizes in X3, rerun X3, then rerun this cell.
import json, importlib, torch, models_x
importlib.reload(models_x)
C = json.load(open("/kaggle/working/run_config_x.json"))
for arch, cfg in C["models"].items():
    try:
        with torch.device("meta"):
            m = models_x.build_model(arch, vocab_size=16000, **cfg)
        n = sum(p.numel() for p in m.parameters())
        extra = ""
        if arch == "moe":
            experts = sum(p.numel() for name, p in m.named_parameters() if ".experts." in name)
            n_exp = cfg["n_experts"]
            extra = f" | active per token ~{(n - experts + experts / n_exp)/1e6:.1f}M"
        print(f"{arch:9s} {n/1e6:6.1f}M total  {'OK' if 50e6 <= n <= 300e6 else 'OUT OF RANGE (need 50M-300M)'}{extra}")
    except Exception as e:
        print(f"{arch:9s} ERROR: {type(e).__name__}: {e}")

## X5c. CPU smoke test (tiny models, ~1–2 min on CPU) — checks that your code runs end to end

In [ ]:
# CPU smoke test — works WITHOUT a GPU (run it in a CPU session while session 2 is still training).
# Builds TINY versions of your models, trains 30 steps on real data and generates 2 short continuations.
import json, importlib, time, torch, numpy as np
import lm_common_x as lmc, models_x
importlib.reload(models_x)

ARCHS_TO_TEST = ["diffusion", "moe"]      # e.g. ["diffusion"] while LexMoE is still a TODO
TINY = {
    "diffusion": {"d_model": 64, "n_layers": 2, "n_heads": 4, "max_len": 256, "dropout": 0.0},
    "moe":       {"d_model": 64, "n_layers": 2, "n_heads": 4, "d_ff": 128, "n_experts": 4, "max_len": 256, "dropout": 0.0},
}
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = lmc.Tok()
data = np.memmap(lmc.find_input("train.bin"), dtype=np.uint16, mode="r")
vp = json.load(open(lmc.find_input("val_prompts.json")))
prompts = [vp[k]["text"] for k in list(vp)[:2]]
for arch in ARCHS_TO_TEST:
    torch.manual_seed(0)
    m = models_x.build_model(arch, vocab_size=tok.V, **TINY[arch]).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    t0, losses = time.time(), []
    for step in range(40):
        seq = lmc.get_batch(data, 8, 128, dev)
        loss = lmc.compute_loss(m, seq, tok.bos, mask_id=tok.mask, aux_coef=0.01)
        loss.backward(); opt.step(); opt.zero_grad()
        losses.append(loss.item())
    aux = getattr(m, "aux_loss", None)
    first5, last5 = np.mean(losses[:5]), np.mean(losses[-5:])
    verdict = "falling - OK" if last5 < first5 else "NOT falling - tell me"
    print(f"{arch:9s} loss (avg of first 5 -> last 5 steps) {first5:.2f} -> {last5:.2f} [{verdict}]"
          + (f" | aux {aux.item():.3f}" if aux is not None else "") + f" | {time.time()-t0:.0f}s")
    if arch == "diffusion":
        out = lmc.generate_diffusion(m, tok, prompts, dev, gen_tokens=48, steps=8, blocks=2, temperature=1.0,
                                     remask="random", target_words=20, log_every=0)
    else:
        out = lmc.generate(m, tok, prompts, dev, temperature=0.8, top_p=0.9, target_words=20, max_new=60, log_every=0)
    print("   words:", [len(o.split()) for o in out], "| sample:", out[0][:100])
print("Smoke test finished — numbers are meaningless at this size; it only checks that everything runs.")

## X6. GPU size/speed check (runs automatically at the start of Save & Run All)

In [ ]:
import json, time, importlib, torch, numpy as np
import lm_common_x as lmc, models_x
importlib.reload(models_x)

C = json.load(open("/kaggle/working/run_config_x.json")); T_ = C["train"]
tok = lmc.Tok()
data = np.memmap(lmc.find_input("train.bin"), dtype=np.uint16, mode="r")
print(f"vocab {tok.V} | ln(V) = {np.log(tok.V):.2f}  (diffusion's first loss is lower: it is weighted by 1/t)")
for arch in ALL_ARCHS:
    m = models_x.build_model(arch, vocab_size=tok.V, **C["models"][arch]).cuda()
    n = sum(p.numel() for p in m.parameters())
    opt, scaler = torch.optim.AdamW(m.parameters(), lr=1e-4), torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    for i in range(13):
        if i == 3:
            torch.cuda.synchronize(); t0 = time.time()
        seq = lmc.get_batch(data, T_["batch_size"], T_["block_size"], "cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            loss = lmc.compute_loss(m, seq, tok.bos, mask_id=tok.mask, aux_coef=C["moe_aux_coef"])
        if i == 0:
            first_loss = loss.item()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    tps = T_["batch_size"] * T_["block_size"] / ((time.time() - t0) / 10)
    print(f"{arch:9s} {n/1e6:6.1f}M params [{'OK' if 50e6 <= n <= 300e6 else 'OUT OF RANGE'}] | "
          f"{tps:,.0f} tok/s | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB | "
          f"~{tps * C['hours_per_model'] * 3600 / 1e6:,.0f}M tokens in budget | loss {first_loss:.2f} -> {loss.item():.2f}")
    del m, opt; torch.cuda.empty_cache()

## X7. Training script (trains for `hours_per_model`, keeps the best checkpoint, writes validation + test predictions and the zip)

In [ ]:
%%writefile train_x.py
import argparse, json, math, os, random, time
import numpy as np, torch
from lm_common_x import (Tok, find_input, get_batch, compute_loss, eval_loss, generate_any, write_submission)
import models_x

ap = argparse.ArgumentParser()
ap.add_argument("--arch", required=True)
ap.add_argument("--gpu", type=int, default=0)
args = ap.parse_args()

C = json.load(open("/kaggle/working/run_config_x.json"))
T_, arch = C["train"], args.arch
dev = f"cuda:{args.gpu}"; torch.cuda.set_device(dev)
random.seed(C["seed"]); np.random.seed(C["seed"]); torch.manual_seed(C["seed"])
OUT = f"/kaggle/working/runs/{arch}"; os.makedirs(OUT, exist_ok=True)
t_start = time.time()
log = open(f"{OUT}/log.jsonl", "a")
def say(**kw):
    kw["t"] = round(time.time() - t_start); log.write(json.dumps(kw) + "\n"); log.flush()
    print(arch, kw, flush=True)

tok = Tok()
train_data = np.memmap(find_input("train.bin"), dtype=np.uint16, mode="r")
val_data = np.memmap(find_input("val.bin"), dtype=np.uint16, mode="r")
model = models_x.build_model(arch, vocab_size=tok.V, **C["models"][arch]).to(dev)
n_params = sum(p.numel() for p in model.parameters())
assert 50e6 <= n_params <= 300e6, f"{arch}: {n_params/1e6:.1f}M params is outside 50M-300M"
block = T_["block_size"]
assert model.max_len >= block
aux_coef = C["moe_aux_coef"]

decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": T_["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}], lr=T_["lr"], betas=tuple(T_["betas"]))
scaler = torch.amp.GradScaler("cuda")
budget = C["hours_per_model"] * 3600

def lr_at(step, frac):                     # linear warm-up, then cosine over the TIME budget
    if step < T_["warmup_steps"]:
        return T_["lr"] * (step + 1) / T_["warmup_steps"]
    return T_["min_lr"] + 0.5 * (T_["lr"] - T_["min_lr"]) * (1 + math.cos(math.pi * min(frac, 1.0)))

def save(path, step, vl):
    torch.save({"model": model.state_dict(), "arch": arch, "cfg": C["models"][arch],
                "step": step, "val_loss": vl}, path)

def evaluate():
    return eval_loss(model, val_data, T_["batch_size"], block, dev, tok.bos, mask_id=tok.mask)

say(event="start", params=n_params, kind=model.kind)
step, tokens, best = 0, 0, float("inf")
model.train()
while (frac := (time.time() - t_start) / budget) < 1:
    lr = lr_at(step, frac)
    for g in opt.param_groups:
        g["lr"] = lr
    tot = 0.0
    for _ in range(T_["grad_accum"]):
        seq = get_batch(train_data, T_["batch_size"], block, dev)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = compute_loss(model, seq, tok.bos, mask_id=tok.mask, aux_coef=aux_coef) / T_["grad_accum"]
        scaler.scale(loss).backward(); tot += loss.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), T_["grad_clip"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    step += 1; tokens += T_["batch_size"] * T_["grad_accum"] * block
    if step % T_["log_every"] == 0:
        extra = {}
        if getattr(model, "aux_loss", None) is not None:
            extra["aux"] = round(float(model.aux_loss), 4)        # 1.0 = perfectly balanced experts
        say(event="train", step=step, loss=round(tot, 4), lr=lr, tokens=tokens, **extra)
    if step % T_["eval_every"] == 0:
        vl = evaluate()
        say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
        if vl < best:
            best = vl; save(f"{OUT}/best.pt", step, vl)

vl = evaluate()
say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
if vl < best:
    best = vl; save(f"{OUT}/best.pt", step, vl)

# ---- generate with the best checkpoint ----
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev)["model"])
del opt; torch.cuda.empty_cache()
vp = json.load(open(find_input("val_prompts.json"))); vids = list(vp)
preds = generate_any(model, tok, [vp[i]["text"] for i in vids], dev, C)
write_submission(dict(zip(vids, preds)), vids, f"{OUT}/val_prediction.json")
tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
preds = generate_any(model, tok, [tp[i]["text"] for i in tids], dev, C)
write_submission(dict(zip(tids, preds)), tids, f"{OUT}/prediction.json", f"{OUT}/submission_{arch}.zip")
say(event="done", best_val_loss=best, steps=step, tokens=tokens)

## X8. Launch: one model per GPU in parallel; prints progress every 2 min

In [ ]:
import subprocess, sys, time, torch

n_gpu = torch.cuda.device_count()
assert n_gpu >= 1
waves = [SESSION_ARCHS[i:i + n_gpu] for i in range(0, len(SESSION_ARCHS), n_gpu)]   # 2 GPUs -> one wave
for wave in waves:
    procs = []
    for gpu, arch in enumerate(wave):
        out = open(f"/kaggle/working/{arch}.out", "w")
        procs.append((arch, subprocess.Popen([sys.executable, "train_x.py", "--arch", arch, "--gpu", str(gpu)],
                                             stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working")))
    while any(p.poll() is None for _, p in procs):
        time.sleep(120)
        for arch, _ in procs:
            lines = open(f"/kaggle/working/{arch}.out").read().strip().splitlines()
            print(time.strftime("%H:%M"), "|", lines[-1][:220] if lines else "(no output yet)", flush=True)
    for arch, p in procs:
        print(f"{arch}: exit code {p.returncode}")
        if p.returncode != 0:
            print(open(f"/kaggle/working/{arch}.out").read()[-3000:])

## X9. Loss curves (+ MoE load balance)

In [ ]:
import json, glob
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for path in sorted(glob.glob("/kaggle/working/runs/*/log.jsonl")):
    arch = path.split("/")[-2]
    rows = [json.loads(l) for l in open(path)]
    tr = [r for r in rows if r["event"] == "train"]; ev = [r for r in rows if r["event"] == "eval"]
    ax[0].plot([r["tokens"] / 1e6 for r in tr], [r["loss"] for r in tr], label=arch, alpha=.8)
    ax[1].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-", label=arch)
    if any("aux" in r for r in tr):
        ax[2].plot([r["tokens"] / 1e6 for r in tr if "aux" in r], [r["aux"] for r in tr if "aux" in r], label=arch)
ax[2].axhline(1.0, ls="--", c="gray", lw=1)
for a, t in zip(ax, ["Training loss", "Validation loss", "MoE load-balancing loss (1.0 = balanced)"]):
    a.set(title=t, xlabel="Training tokens (M)"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curves_extra.png", dpi=200); plt.show()

## X10. Official local evaluation on your validation set

In [ ]:
%%writefile evaluate_phase1.py
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)

In [ ]:
!pip install -q bert-score
import os, lm_common_x as lmc
VAL_TRUTH = lmc.find_input("val_truth.json")
for arch in sorted(os.listdir("/kaggle/working/runs")):
    pred = f"/kaggle/working/runs/{arch}/val_prediction.json"
    if os.path.exists(pred):
        print("=====", arch)
        !python evaluate_phase1.py --truth {VAL_TRUTH} --prediction {pred}

## X11. Check the submission zips before uploading to Codabench

In [ ]:
import zipfile, json, glob
test_ids = set(json.load(open("/kaggle/working/test.json")))
for z in sorted(glob.glob("/kaggle/working/runs/*/submission_*.zip")):
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    wc = [len(v["text"].split()) for v in pred.values()]
    ok = names == ["prediction.json"] and set(pred) == test_ids and all(v["text"].strip() for v in pred.values())
    print(f"{z}: {'OK' if ok else 'PROBLEM'} | files={names} | ids={len(pred)} | "
          f"words min/mean/max = {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)}")

## X12. LexMoE — expert routing by court type (for the report)
Set `SOURCE_COUNTS` from the data-prep notebook's **A9** output first. Interpreting the table and heatmap is your job for the report: do some experts serve mostly statutes, or mostly High Court orders?

In [ ]:
# Which expert does each court type use? Runs the trained LexMoE on the validation prompts, which are stored
# in source order. Copy the per-source counts from the data-prep notebook's A9 output ("validation examples
# per source: {...}") — SAME ORDER, numbers must add up to the number of validation prompts.
import json, torch, numpy as np, matplotlib.pyplot as plt
import lm_common_x as lmc, models_x

SOURCE_COUNTS = {"supreme_court": 120, "high_courts": 100, "tribunals": 55, "acts": 12}   # <-- from A9 output

dev = "cuda:0"
tok = lmc.Tok()
ck = torch.load(lmc.find_input("moe/best.pt"), map_location=dev)
model = models_x.build_model("moe", vocab_size=tok.V, **ck["cfg"]).to(dev).eval()
vp = json.load(open(lmc.find_input("val_prompts.json"))); vids = list(vp)
assert sum(SOURCE_COUNTS.values()) == len(vids), f"counts add to {sum(SOURCE_COUNTS.values())}, prompts = {len(vids)}"
sources = [s for s, n in SOURCE_COUNTS.items() for _ in range(n)]
moe_layers = [m for m in model.modules() if hasattr(m, "last_choice")]
E = moe_layers[0].n_experts
counts = np.zeros((len(moe_layers), len(SOURCE_COUNTS), E))
names = list(SOURCE_COUNTS)
with torch.no_grad():
    for k, i in enumerate(vids):
        x = torch.tensor([tok.encode(vp[i]["text"])[-model.max_len:]], device=dev)
        with torch.autocast("cuda", dtype=torch.float16):
            model(x)
        s = names.index(sources[k])
        for l, layer in enumerate(moe_layers):
            counts[l, s] += np.bincount(layer.last_choice.flatten().cpu().numpy(), minlength=E)
share = counts / np.maximum(counts.sum(-1, keepdims=True), 1)   # (layer, source, expert): fraction of tokens

for l in range(len(moe_layers)):
    print(f"layer {l}: " + " | ".join(f"{n[:6]} " + " ".join(f"{v:.2f}" for v in share[l, s]) for s, n in enumerate(names)))
fig, axes = plt.subplots(1, len(moe_layers), figsize=(2.4 * len(moe_layers), 3.2), sharey=True)
for l, a in enumerate(np.atleast_1d(axes)):
    a.imshow(share[l], vmin=0, vmax=1, cmap="viridis")
    a.set_title(f"layer {l}"); a.set_xticks(range(E)); a.set_xlabel("expert")
    a.set_yticks(range(len(names))); a.set_yticklabels(names)
plt.suptitle("Share of each court type's tokens routed to each expert"); plt.tight_layout()
plt.savefig("/kaggle/working/moe_routing_by_source.png", dpi=200); plt.show()
np.save("/kaggle/working/moe_routing_share.npy", share)

## X13. (Optional) Diffusion sampling comparison on the validation set
Sampling settings change a diffusion model's output a lot. If the X10 scores or the samples look poor (for example lots of
repeated punctuation), add candidates to `SETTINGS` here, e.g. a different `temperature`, `remask`, `steps` or `blocks`,
and compare them by the official scores. No retraining needed. Each extra test submission made this way is genuinely different.

In [ ]:
# Compare diffusion sampling settings on the validation set (no retraining). Each setting takes a few minutes.
# Pick the best by the official scores, then (optionally) write a new test submission with it.
import json, torch, lm_common_x as lmc, models_x
SETTINGS = {                       # YOUR candidates — the first one is the setting used during the main run
    "main": {},
}
C = json.load(open("/kaggle/working/run_config_x.json"))
dev = "cuda:0"
tok = lmc.Tok()
ck = torch.load(lmc.find_input("diffusion/best.pt"), map_location=dev)
model = models_x.build_model("diffusion", vocab_size=tok.V, **ck["cfg"]).to(dev).eval()
vp = json.load(open(lmc.find_input("val_prompts.json"))); vids = list(vp)
truth = lmc.find_input("val_truth.json")
for name, s in SETTINGS.items():
    cfg = {**C["diff_gen"], **s}
    preds = lmc.generate_diffusion(model, tok, [vp[i]["text"] for i in vids], dev, **cfg)
    path = f"/kaggle/working/sweep_diffusion_{name}.json"
    lmc.write_submission(dict(zip(vids, preds)), vids, path)
    print("=====", name, cfg)
    !python evaluate_phase1.py --truth {truth} --prediction {path}

# To write a test submission with the setting you chose:
# best = {**C["diff_gen"], **SETTINGS["main"]}; tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
# preds = lmc.generate_diffusion(model, tok, [tp[i]["text"] for i in tids], dev, **best)
# lmc.write_submission(dict(zip(tids, preds)), tids, "/kaggle/working/prediction.json",
#                      "/kaggle/working/submission_diffusion_v2.zip")